[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C61_Detection_Practice_Interview_Course/02_error_analysis/02_error_analysis.ipynb)

# 02 · 误差分析工程（TIDE 六类分解 / 修复收益 / PR 诊断 / 工作点 / 分层抽样）

目标：把「mAP = 0.60，下一步做什么」这个问题，变成一串**可以算出来的数字**。

本 notebook 你会亲手实现：
1. **可控误差的合成 TSR 评测集** —— 用 `IoU = (w-dx)/(w+dx)` 精确注入指定 IoU 的定位误差
2. **VOC 全点 AP / mAP**（单调包络 + 面积）
3. **完整的 TIDE 六类误差分解**：Cls / Loc / Both / Dupe / Bkg / Miss
4. **六个 oracle 与修复收益 ΔmAP** —— 「修好每一类能涨多少」，并验证 **ΔAP 不可加**
5. **逐类 AP + $(C{+}1)\times(C{+}1)$ 混淆矩阵**，自动找出最容易互相错分的标志对
6. **PR 曲线诊断器**：从曲线形状读出「管线 bug / 召回天花板 / 尾部崩塌 / 健康」
7. **工作点求解**：给定 FP/frame 预算求每类的最优 score 阈值
8. **badcase 分层抽样器**：平方根配额 + 最大余数法 + 按修复收益加权
9. **根因决策树的代码化**

> 心智模型：**mAP 告诉你「有多差」，误差分解告诉你「修哪个最划算」。**

## 1 · 工具：IoU 与「精确注入指定 IoU」

后面所有实验都要能**按需造出 IoU 恰好等于 0.32 的框**。用水平平移就能做到闭式解：
同尺寸的框水平平移 $dx$ 后，交 $=(w-dx)h$、并 $=2wh-(w-dx)h$，于是

$$\text{IoU} = \frac{w-dx}{w+dx} \quad\Longrightarrow\quad dx = w\cdot\frac{1-\text{IoU}}{1+\text{IoU}}$$

顺带记住那个**面试常考的数字**：$8\times8$ 的框平移 2 px，IoU $= 6/10 = 0.6$；沿对角线平移 2 px 则是 $36/92 = 0.391$。

In [ ]:
import numpy as np
from collections import defaultdict, Counter

rng = np.random.default_rng(7)

CLASSES = ['限速30', '限速60', '限速80', '停车让行', '禁止左转', '注意行人', '解除限速', '指路牌']
NC = len(CLASSES)


def iou_matrix(a, b):
    # a: (N,4) xyxy, b: (M,4) xyxy -> (N,M)
    a = np.asarray(a, float).reshape(-1, 4)
    b = np.asarray(b, float).reshape(-1, 4)
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    aa = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    bb = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / np.maximum(aa[:, None] + bb[None, :] - inter, 1e-9)


def shift_for_iou(w, target_iou):
    # 水平平移量，使平移后的同尺寸框与原框 IoU 恰好等于 target_iou
    return w * (1.0 - target_iou) / (1.0 + target_iou)


base = np.array([[100., 100., 140., 140.]])          # 40x40 的框
for t in [0.9, 0.6, 0.5, 0.3, 0.15]:
    dx = shift_for_iou(40.0, t)
    got = iou_matrix(base, base + np.array([dx, 0, dx, 0]))[0, 0]
    assert abs(got - t) < 1e-9, (t, got)
print('✅ shift_for_iou 精确：可以按需造出任意 IoU 的定位误差')

# 小目标对位移的敏感性（面试常考）
small = np.array([[0., 0., 8., 8.]])
print('8x8 框水平移 2px 的 IoU =', round(float(iou_matrix(small, small + [2, 0, 2, 0])[0, 0]), 4), '= 6/10')
print('8x8 框对角移 2px 的 IoU =', round(float(iou_matrix(small, small + [2, 2, 2, 2])[0, 0]), 4), '= 36/92')
assert abs(float(iou_matrix(small, small + [2, 2, 2, 2])[0, 0]) - 36 / 92) < 1e-12
big = np.array([[0., 0., 64., 64.]])
print('64x64 框对角移 2px 的 IoU =', round(float(iou_matrix(big, big + [2, 2, 2, 2])[0, 0]), 4))
print('👉 同样 2 px 的误差，小框跌破 0.5 阈值，大框几乎无损 —— 这是 Loc 错误的物理来源')

## 2 · 合成一个可控的 TSR 评测集

**合成规则**（每一条都对应真实检测器的一种行为）：
- 200 帧 1920×1080，每帧 1–4 块标志；类别按长尾先验采样（限速类多、停车让行少）
- 标志边长服从对数正态（中位数 30 px，从 10 px 到 130 px）—— TSR 的真实尺寸分布
- 每个 GT 按固定概率落入五种结局之一：**TP / Miss / Loc / Cls / Both**
  - `Loc`：同类，IoU 精确落在 [0.18, 0.46)（跌破 0.5 阈值）
  - `Cls`：框几乎完美，但类别被换成**易混淆的那个**（限速60↔80、禁左→解除限速）
  - `Both`：框歪 + 类错
- TP 有 12% 概率再生一个**重复框**（IoU 0.55–0.8，分数打折）→ Dupe
- 每帧额外 7 个**背景误检**，分数偏低但有长尾（模拟广告牌/车身贴纸偶尔拿到高分）

> 这样我们**事先就知道地面真相**，可以验证分解器是否把每类错误都找了回来。

In [ ]:
# 易混淆映射：只有限速类家族内部会互相错分（映射到自己 = 没有「双胞胎」）
CONFUSE = {0: 2, 1: 2, 2: 1, 3: 3, 4: 4, 5: 5, 6: 1, 7: 7}
CLS_PRIOR = np.array([0.10, 0.24, 0.20, 0.05, 0.09, 0.08, 0.06, 0.18])
CLS_PRIOR = CLS_PRIOR / CLS_PRIOR.sum()

N_IMG, W, H = 200, 1920, 1080
P_MISS, P_LOC, P_CLS, P_BOTH = 0.09, 0.10, 0.16, 0.05       # 其余为 TP
P_DUPE, N_BKG_PER_IMG = 0.12, 7

g_img, g_cls, g_box = [], [], []
d_img, d_cls, d_box, d_score = [], [], [], []

for im in range(N_IMG):
    for _ in range(int(rng.integers(1, 5))):
        c = int(rng.choice(NC, p=CLS_PRIOR))
        s = float(np.clip(np.exp(rng.normal(np.log(30), 0.55)), 10, 130))    # 标志边长 px
        cx, cy = float(rng.uniform(120, W - 120)), float(rng.uniform(80, H * 0.62))
        gb = np.array([cx - s / 2, cy - s / 2, cx + s / 2, cy + s / 2])
        g_img.append(im); g_cls.append(c); g_box.append(gb)

        u = rng.random()
        if u < P_MISS:                                        # 【Miss】完全没输出
            continue
        elif u < P_MISS + P_LOC:                              # 【Loc】类对框歪
            dx = shift_for_iou(s, float(rng.uniform(0.18, 0.46)))
            db, dc = gb + np.array([dx, 0, dx, 0]), c
            sc = 0.28 + 0.55 * rng.beta(2.0, 2.0)
        elif u < P_MISS + P_LOC + P_CLS:                      # 【Cls】框对类错（且很自信）
            db = gb + rng.normal(0, 0.02 * s, 4)
            dc = CONFUSE[c]                                   # 没有双胞胎的类 -> 仍是 TP
            sc = 0.40 + 0.52 * rng.beta(2.6, 1.4)
        elif u < P_MISS + P_LOC + P_CLS + P_BOTH:             # 【Both】都错
            dx = shift_for_iou(s, float(rng.uniform(0.15, 0.45)))
            db = gb + np.array([dx, 0, dx, 0])
            dc = CONFUSE[c]                                   # 没有双胞胎的类 -> 退化为 Loc
            sc = 0.25 + 0.5 * rng.beta(2.0, 2.2)
        else:                                                 # 【TP】
            db, dc = gb + rng.normal(0, 0.012 * s, 4), c
            sc = 0.45 + 0.5 * rng.beta(2.8, 1.2)
            if rng.random() < P_DUPE:                         # 【Dupe】重复框
                dx2 = shift_for_iou(s, float(rng.uniform(0.55, 0.8)))
                d_img.append(im); d_cls.append(c)
                d_box.append(gb + np.array([dx2, 0, dx2, 0]))
                d_score.append(sc * float(rng.uniform(0.45, 0.9)))
        d_img.append(im); d_cls.append(dc); d_box.append(db); d_score.append(min(sc, 0.995))

    for _ in range(N_BKG_PER_IMG):                            # 【Bkg】背景误检
        s = float(np.exp(rng.normal(np.log(34), 0.6)))
        cx, cy = float(rng.uniform(60, W - 60)), float(rng.uniform(40, H - 40))
        d_img.append(im); d_cls.append(int(rng.choice(NC, p=CLS_PRIOR)))
        d_box.append(np.array([cx - s / 2, cy - s / 2, cx + s / 2, cy + s / 2]))
        d_score.append(0.05 + 0.88 * rng.beta(1.5, 3.4))

GT = dict(img=np.array(g_img), cls=np.array(g_cls), box=np.array(g_box))
DT = dict(img=np.array(d_img), cls=np.array(d_cls), box=np.array(d_box), score=np.array(d_score))
print(f'GT {len(GT["img"])} 个目标 / {N_IMG} 帧;  检测 {len(DT["img"])} 个框')
print('每类 GT 数:', {CLASSES[c]: int((GT['cls'] == c).sum()) for c in range(NC)})
assert len(GT['img']) > 400 and len(DT['img']) > 1500

## 3 · VOC 全点 AP 与 mAP

判定流程（C18 模块 03 的复习）：**按分数降序 → 每个检测取同图同类 IoU 最大的 GT →
IoU ≥ 0.5 且该 GT 未被占用则 TP，否则 FP → 累积 P/R → 单调包络 → 求面积**。

注意「IoU ≥ 0.5 但 GT 已被更高分的框占用 ⇒ FP」这一条——**它就是 Dupe 的来源**。

In [ ]:
def match_class(dets, gts, c, iou_thr=0.5):
    # 返回该类别下按分数降序的 (scores, is_tp, best_gt_idx, n_gt)
    di = np.where(dets['cls'] == c)[0]
    gi = np.where(gts['cls'] == c)[0]
    n_gt = len(gi)
    if len(di) == 0:
        return np.zeros(0), np.zeros(0, bool), np.zeros(0, int), n_gt
    order = di[np.argsort(-dets['score'][di], kind='stable')]
    gt_by_img = defaultdict(list)
    for j in gi:
        gt_by_img[int(gts['img'][j])].append(j)
    matched, is_tp, best = set(), np.zeros(len(order), bool), np.full(len(order), -1)
    for r, i in enumerate(order):
        cand = gt_by_img.get(int(dets['img'][i]), [])
        if not cand:
            continue
        ious = iou_matrix(dets['box'][i:i + 1], gts['box'][cand])[0]
        k = int(np.argmax(ious))
        if ious[k] >= iou_thr:
            best[r] = cand[k]
            if cand[k] not in matched:                 # GT 未被占 -> TP；已被占 -> FP（= Dupe）
                matched.add(cand[k]); is_tp[r] = True
    return dets['score'][order], is_tp, best, n_gt


def voc_ap(rec, prec):
    # 单调包络 + 面积（VOC 全点插值）
    if len(rec) == 0:
        return 0.0
    mrec = np.concatenate([[0.0], rec, [rec[-1]]])
    mpre = np.concatenate([[0.0], prec, [0.0]])
    for i in range(len(mpre) - 2, -1, -1):
        mpre[i] = max(mpre[i], mpre[i + 1])
    idx = np.where(mrec[1:] != mrec[:-1])[0]
    return float(np.sum((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1]))


def class_pr(dets, gts, c, iou_thr=0.5):
    sc, tp, _, n_gt = match_class(dets, gts, c, iou_thr)
    if len(sc) == 0 or n_gt == 0:
        return np.zeros(0), np.zeros(0)
    ctp = np.cumsum(tp.astype(float)); cfp = np.cumsum((~tp).astype(float))
    return ctp / n_gt, ctp / np.maximum(ctp + cfp, 1e-12)


def evaluate(dets, gts, iou_thr=0.5):
    aps = {}
    for c in range(NC):
        n_gt = int((gts['cls'] == c).sum())
        if n_gt == 0:
            continue
        rec, prec = class_pr(dets, gts, c, iou_thr)
        aps[c] = voc_ap(rec, prec)
    return float(np.mean(list(aps.values()))), aps


# 手算校验：一个 GT、两个检测（高分 TP + 低分 Dupe）
tiny_g = dict(img=np.array([0]), cls=np.array([0]), box=np.array([[0., 0., 10., 10.]]))
tiny_d = dict(img=np.array([0, 0]), cls=np.array([0, 0]),
              box=np.array([[0., 0., 10., 10.], [1., 0., 11., 10.]]), score=np.array([0.9, 0.8]))
m, a = evaluate(tiny_d, tiny_g)
assert abs(a[0] - 1.0) < 1e-12, a          # 第 1 个就命中，recall 直接到 1，AP=1
print('手算校验 AP =', a[0], '（第一个检测就命中，后面的 Dupe 不再增加 recall）')

mAP0, aps0 = evaluate(DT, GT)
print(f'\n=== baseline mAP@0.5 = {mAP0:.4f} ===')
print(f'{"类别":<10s}{"AP":>8s}{"GT 数":>8s}')
for c in range(NC):
    print(f'{CLASSES[c]:<10s}{aps0[c]:>8.3f}{int((GT["cls"] == c).sum()):>8d}')
assert 0.4 < mAP0 < 0.8
print('\n👉 拿到这个 0.60，你现在还是不知道下一步该做什么。继续往下。')

## 4 · TIDE 六类误差分解

对每个非 TP 的检测算两个数：
- $u_{\text{same}}$：与**同类** GT 的最大 IoU
- $u_{\text{other}}$：与**异类** GT 的最大 IoU

然后走判定树（顺序不能变）：

```
u_same >= 0.5              -> Dupe   （能匹配，但 GT 被更高分的框占了）
0.1 <= u_same < 0.5        -> Loc    （类对框歪）
u_other >= 0.5             -> Cls    （框对类错）
0.1 <= u_other < 0.5       -> Both
否则                        -> Bkg    （周围什么都没有）
```

**Miss** 单独算：所有「没有任何检测以 IoU ≥ 0.1 靠近过」的 GT。
这个定义保证了 Miss 与 Cls/Loc **不重复计数**——被 Cls 错误覆盖的 GT 不算 Miss。

In [ ]:
def tide_classify(dets, gts, tf=0.5, tb=0.1):
    # 返回 (每个检测的错误标签, 归因到的 GT 下标, 每个 GT 是否 Miss, u_same, u_other)
    n_d = len(dets['img'])
    label = np.array(['Bkg'] * n_d, dtype=object)
    tgt = np.full(n_d, -1)
    u_same = np.zeros(n_d); u_other = np.zeros(n_d)

    for c in range(NC):                                    # ① 先做匹配，确定 TP
        di = np.where(dets['cls'] == c)[0]
        if len(di) == 0:
            continue
        order = di[np.argsort(-dets['score'][di], kind='stable')]
        _, is_tp, best, _ = match_class(dets, gts, c, tf)
        for r, i in enumerate(order):
            if is_tp[r]:
                label[i] = 'TP'; tgt[i] = best[r]

    gt_by_img = defaultdict(list)
    for j in range(len(gts['img'])):
        gt_by_img[int(gts['img'][j])].append(j)

    for i in range(n_d):                                   # ② 非 TP 的走判定树
        cand = np.array(gt_by_img.get(int(dets['img'][i]), []), dtype=int)
        j_s = j_o = -1
        if len(cand):
            same = cand[gts['cls'][cand] == dets['cls'][i]]
            other = cand[gts['cls'][cand] != dets['cls'][i]]
            if len(same):
                v = iou_matrix(dets['box'][i:i + 1], gts['box'][same])[0]
                k = int(np.argmax(v)); u_same[i], j_s = float(v[k]), int(same[k])
            if len(other):
                v = iou_matrix(dets['box'][i:i + 1], gts['box'][other])[0]
                k = int(np.argmax(v)); u_other[i], j_o = float(v[k]), int(other[k])
        if label[i] == 'TP':
            continue
        if u_same[i] >= tf:
            label[i], tgt[i] = 'Dupe', j_s
        elif u_same[i] >= tb:
            label[i], tgt[i] = 'Loc', j_s
        elif u_other[i] >= tf:
            label[i], tgt[i] = 'Cls', j_o
        elif u_other[i] >= tb:
            label[i], tgt[i] = 'Both', j_o
        else:
            label[i] = 'Bkg'

    covered = np.zeros(len(gts['img']), bool)              # ③ Miss：没被任何框靠近过的 GT
    for i in range(n_d):
        cand = gt_by_img.get(int(dets['img'][i]), [])
        if not cand:
            continue
        v = iou_matrix(dets['box'][i:i + 1], gts['box'][cand])[0]
        for k, j in enumerate(cand):
            if v[k] >= tb:
                covered[j] = True
    return label, tgt, ~covered, u_same, u_other


LAB, TGT, MISS, U_SAME, U_OTHER = tide_classify(DT, GT)
CNT = Counter(LAB.tolist()); CNT['Miss'] = int(MISS.sum())
ERR_TYPES = ['Cls', 'Loc', 'Both', 'Dupe', 'Bkg', 'Miss']

print(f'{"类型":<8s}{"数量":>8s}{"占全部错误":>12s}')
n_err = sum(CNT[e] for e in ERR_TYPES)
for e in ERR_TYPES:
    print(f'{e:<8s}{CNT[e]:>8d}{CNT[e] / n_err:>11.1%}')
print(f'{"TP":<8s}{CNT["TP"]:>8d}')

assert CNT['TP'] + CNT['Loc'] + CNT['Cls'] + CNT['Both'] + CNT['Dupe'] + CNT['Bkg'] == len(DT['img'])
assert CNT['Bkg'] > 10 * CNT['Cls'], '合成时背景误检就是最多的'
print(f'\n⚠️  Bkg 的数量是 Cls 的 {CNT["Bkg"] / CNT["Cls"]:.0f} 倍。凭数量做决策的话，你会去治背景误检。')
print('   下一节会证明这是错的。')

## 5 · 修复收益：ΔmAP 才是决策依据

六个 **oracle**（先知）各修好一类错误，其余不动，重算 mAP：

| oracle | 动作 |
|---|---|
| `Loc`  | 把 Loc 错误的框**吸附到**它归因的 GT 上（分数与排序不变）|
| `Cls`  | 把 Cls 错误的**预测类别改对**（框不动）|
| `Both` | 类别与框一起修 |
| `Dupe` | **删掉**重复框 |
| `Bkg`  | **删掉**背景误检 |
| `Miss` | 把「完全没被看见」的 GT **从评测集里移除**（改的是 $G$ 不是 $D$）|

$$\Delta\text{AP}_i = \text{mAP}\bigl(\mathcal{O}_i(D),\,G\bigr) - \text{mAP}(D,\,G)$$

In [ ]:
def apply_oracle(kind, dets, gts, lab, tgt, miss):
    d = {k: v.copy() for k, v in dets.items()}
    g = {k: v.copy() for k, v in gts.items()}
    if kind == 'Loc':
        m = (lab == 'Loc'); d['box'][m] = g['box'][tgt[m]]
    elif kind == 'Cls':
        m = (lab == 'Cls'); d['cls'][m] = g['cls'][tgt[m]]
    elif kind == 'Both':
        m = (lab == 'Both'); d['cls'][m] = g['cls'][tgt[m]]; d['box'][m] = g['box'][tgt[m]]
    elif kind in ('Dupe', 'Bkg'):
        keep = (lab != kind); d = {k: v[keep] for k, v in d.items()}
    elif kind == 'Miss':
        g = {k: v[~miss] for k, v in g.items()}
    return d, g


GAINS, PER100 = {}, {}
print(f'{"类型":<7s}{"数量":>7s}{"修好后 mAP":>12s}{"ΔmAP(pp)":>11s}{"每100个错的ΔmAP":>17s}')
for kind in ERR_TYPES:
    d2, g2 = apply_oracle(kind, DT, GT, LAB, TGT, MISS)
    m2, _ = evaluate(d2, g2)
    GAINS[kind] = m2 - mAP0
    PER100[kind] = GAINS[kind] * 100 / max(CNT[kind], 1) * 100
    print(f'{kind:<7s}{CNT[kind]:>7d}{m2:>12.4f}{GAINS[kind] * 100:>+11.2f}{PER100[kind]:>17.3f}')

for k in ['Dupe', 'Bkg', 'Miss']:
    assert GAINS[k] >= -1e-9, (k, GAINS[k])          # 删 FP / 删未覆盖 GT 只可能不变差
assert GAINS['Cls'] > GAINS['Bkg'], '数量最多的 Bkg 反而最不值钱'
assert PER100['Cls'] > 50 * PER100['Bkg']

print(f'\n六项 ΔmAP 之和 = {sum(GAINS.values()) * 100:.2f} pp')
print(f'离满分的总差距   = {(1 - mAP0) * 100:.2f} pp')
assert sum(GAINS.values()) < 1 - mAP0
print('👉 **ΔAP 不可加**：AP 经过排序/包络/求面积三重非线性，oracle 的效应不满足叠加原理。')
print('   两类错误常常打在同一批 GT 上，第一个 oracle 救回来的，第二个就救不到了。')

In [ ]:
# 决策：ΔmAP / 工程成本 —— 真实世界的排序依据
COST = {'Cls': 3.0, 'Loc': 5.0, 'Both': 5.0, 'Dupe': 0.5, 'Bkg': 4.0, 'Miss': 12.0}   # 人天（估）
print(f'{"类型":<7s}{"ΔmAP(pp)":>10s}{"成本(人天)":>11s}{"ROI = pp/人天":>15s}')
roi = {k: GAINS[k] * 100 / COST[k] for k in ERR_TYPES}
for k in sorted(roi, key=lambda x: -roi[x]):
    print(f'{k:<7s}{GAINS[k] * 100:>+10.2f}{COST[k]:>11.1f}{roi[k]:>15.3f}')
best = max(roi, key=lambda k: roi[k])
print(f'\n👉 按 ΔmAP 排序第一名 = {max(GAINS, key=lambda k: GAINS[k])}')
print(f'   按 ROI  排序第一名 = {best}')
print('   两者不一定相同 —— Miss 的 ΔmAP 不小，但要靠「提分辨率 + 加 P2 层」兑现，代价最高。')
assert roi['Cls'] > roi['Bkg'] and roi['Cls'] > roi['Miss']

## 6 · 逐类 AP 与 $(C{+}1)\times(C{+}1)$ 混淆矩阵

检测的混淆矩阵**必须多一行一列**：
- 最后一**列** = 该类 GT 没被任何检测认领（Miss + 定位失败）
- 最后一**行** = 背景被误检成某类（Bkg）

只画 $C\times C$ 的话，「停车让行召回只有 68%」这种致命信息会被完全隐藏。

In [ ]:
def confusion_matrix(dets, gts, score_thr=0.30, iou_thr=0.5):
    cm = np.zeros((NC + 1, NC + 1), dtype=int)
    keep = np.where(dets['score'] >= score_thr)[0]
    keep = keep[np.argsort(-dets['score'][keep], kind='stable')]      # 类别无关的贪心匹配
    gt_by_img = defaultdict(list)
    for j in range(len(gts['img'])):
        gt_by_img[int(gts['img'][j])].append(j)
    used = set()
    for i in keep:
        cand = [j for j in gt_by_img.get(int(dets['img'][i]), []) if j not in used]
        if cand:
            v = iou_matrix(dets['box'][i:i + 1], gts['box'][cand])[0]
            k = int(np.argmax(v))
            if v[k] >= iou_thr:
                used.add(cand[k])
                cm[int(gts['cls'][cand[k]]), int(dets['cls'][i])] += 1
                continue
        cm[NC, int(dets['cls'][i])] += 1                              # 背景误检
    for j in range(len(gts['img'])):
        if j not in used:
            cm[int(gts['cls'][j]), NC] += 1                           # 未被认领的 GT
    return cm


SCORE_THR = 0.30
CM = confusion_matrix(DT, GT, SCORE_THR)
assert (CM[:NC].sum(1) == np.bincount(GT['cls'], minlength=NC)).all(), '每行之和必须等于该类 GT 数'

hdr = ''.join(f'{c[:4]:>7s}' for c in CLASSES) + f'{"未检出":>8s}'
print(f'混淆矩阵（score >= {SCORE_THR}）  行=真值, 列=预测')
print(f'{"":<10s}{hdr}')
for i in range(NC):
    row = ''.join(f'{CM[i, j]:>7d}' for j in range(NC))
    print(f'{CLASSES[i]:<10s}{row}{CM[i, NC]:>8d}')
row = ''.join(f'{CM[NC, j]:>7d}' for j in range(NC))
print(f'{"背景误检":<10s}{row}{"-":>8s}')

In [ ]:
def top_confused(cm, k=4):
    # 对称化的非对角热点：最容易互相错分的标志对
    out = [(int(cm[i, j] + cm[j, i]), i, j) for i in range(NC) for j in range(i + 1, NC)]
    out.sort(reverse=True)
    return out[:k]


print('最容易互相错分的标志对：')
for n, i, j in top_confused(CM):
    print(f'  {CLASSES[i]} <-> {CLASSES[j]:<8s} {n:>4d} 次')
worst = top_confused(CM, 1)[0]
assert {worst[1], worst[2]} == {1, 2}, '合成时就是限速60↔80 最混'
print('👉 限速 60 ↔ 80 是 TSR 的经典冤家：整体形状颜色完全一致，')
print('   差异只在数字中间那一横 —— 20 px 的框里它只有约 1.6 px 宽。')

print(f'\n{"类别":<10s}{"AP":>7s}{"召回":>8s}{"被误检成它":>12s}{"GT数":>7s}')
for i in range(NC):
    rec_i = 1 - CM[i, NC] / max(CM[i].sum(), 1)
    print(f'{CLASSES[i]:<10s}{aps0[i]:>7.3f}{rec_i:>8.2f}{CM[NC, i]:>12d}{int(CM[i].sum()):>7d}')
print('\n⚠️  只看第一列（AP）会漏掉两件事：该类的召回天花板、该类是不是「误检磁铁」。')

## 7 · PR 曲线诊断：形状即证据

四种形状对应四种病。先算三个关键量（$p(r)$ 取单调包络意义下的 $\max_{\tilde r\ge r} p(\tilde r)$）：

| 量 | 含义 | 读法 |
|---|---|---|
| `p10` = $p(r{=}0.1)$ | 最自信的那批预测的精度 | **< 0.8 ⇒ 管线 bug**，不要调参 |
| `r_max` | 召回天花板 | **< 0.75 ⇒ 有一批 GT 从没被覆盖** |
| `p50`, `p85` | 中段与尾段精度 | `p50 高 + p85 崩` ⇒ 尾部塌陷 |

下面先造四条**手工曲线**当标尺，再用它们量真实模型。

In [ ]:
def p_at_recall(rec, prec, r):
    m = np.asarray(rec) >= r
    return float(np.asarray(prec)[m].max()) if m.any() else 0.0


def pr_shape_stats(rec, prec):
    rec = np.asarray(rec, float); prec = np.asarray(prec, float)
    return dict(r_max=float(rec.max()) if rec.size else 0.0,
                p10=p_at_recall(rec, prec, 0.10),
                p50=p_at_recall(rec, prec, 0.50),
                p85=p_at_recall(rec, prec, 0.85))


def make_curve(kind):
    if kind == 'HEALTHY':          # 平滑衰减
        r = np.linspace(0.02, 0.90, 60); p = 0.99 - 0.35 * (r / 0.9) ** 2
    elif kind == 'TAIL_COLLAPSE':  # 高分段完美，某点后突然崩
        r1 = np.linspace(0.02, 0.70, 40); p1 = np.full_like(r1, 0.97)
        r2 = np.linspace(0.71, 0.93, 40); p2 = np.linspace(0.55, 0.04, 40)
        r, p = np.concatenate([r1, r2]), np.concatenate([p1, p2])
    elif kind == 'DATA_BUG':       # 最自信的预测就在错
        r = np.linspace(0.02, 0.85, 50); p = 0.44 - 0.10 * (r / 0.85)
    else:                          # RECALL_CEILING：曲线中途断掉
        r = np.linspace(0.02, 0.55, 40); p = 0.96 - 0.10 * (r / 0.55)
    return r, p


print(f'{"标尺曲线":<16s}{"r_max":>8s}{"p10":>8s}{"p50":>8s}{"p85":>8s}')
REF = {}
for k in ['HEALTHY', 'TAIL_COLLAPSE', 'DATA_BUG', 'RECALL_CEILING']:
    r, p = make_curve(k); s = pr_shape_stats(r, p); REF[k] = s
    print(f'{k:<16s}{s["r_max"]:>8.2f}{s["p10"]:>8.2f}{s["p50"]:>8.2f}{s["p85"]:>8.2f}')

assert REF['DATA_BUG']['p10'] < 0.80              # 低召回段就不高
assert REF['RECALL_CEILING']['r_max'] < 0.75      # 天花板
assert REF['TAIL_COLLAPSE']['p50'] > 0.90 and REF['TAIL_COLLAPSE']['p85'] < 0.30
assert REF['HEALTHY']['p10'] > 0.95 and REF['HEALTHY']['r_max'] > 0.85
print('\n✅ 三条判据都是可代码化的阈值，不是「看图感觉」。')

In [ ]:
# ① 真实模型的逐类曲线
print(f'{"类别":<10s}{"r_max":>8s}{"p10":>8s}{"p50":>8s}{"p85":>8s}')
base_stats = {}
for c in range(NC):
    s = pr_shape_stats(*class_pr(DT, GT, c)); base_stats[c] = s
    print(f'{CLASSES[c]:<10s}{s["r_max"]:>8.2f}{s["p10"]:>8.2f}{s["p50"]:>8.2f}{s["p85"]:>8.2f}')
print('👉 每一类的 p10 都是 1.00（最自信的预测都对）=> 管线没问题；')
print('   但 r_max 全都不到 0.8 => **召回天花板**。天花板从哪来？下面两个实验各给一半答案。')

# ② 注入一个「类别整体 +1」的管线 bug —— p10 立刻塌掉
DT_BUG = {k: v.copy() for k, v in DT.items()}
DT_BUG['cls'] = (DT_BUG['cls'] + 1) % NC
mAP_bug, _ = evaluate(DT_BUG, GT)
s_bug = pr_shape_stats(*class_pr(DT_BUG, GT, 1))
print(f'\n[bug 版本] 类别整体 +1 后：mAP = {mAP_bug:.4f}, 限速60 的 p10 = {s_bug["p10"]:.2f}')
assert mAP_bug < 0.05 and s_bug['p10'] < 0.80
print('✅ 这就是模块 03 要处理的「mAP 恒为 0」的头号元凶，PR 曲线的左端立刻暴露它。')

# ③ 把 Cls + Loc 两类错误修好，天花板会不会抬起来？
d2, g2 = apply_oracle('Cls', DT, GT, LAB, TGT, MISS)
L2, T2, M2, _, _ = tide_classify(d2, g2)
d3, g3 = apply_oracle('Loc', d2, g2, L2, T2, M2)
mAP_fix, _ = evaluate(d3, g3)
print(f'\n[修好 Cls+Loc] mAP {mAP0:.3f} -> {mAP_fix:.3f}')
print(f'{"类别":<10s}{"r_max 前":>10s}{"r_max 后":>10s}')
for c in range(NC):
    s = pr_shape_stats(*class_pr(d3, g3, c))
    print(f'{CLASSES[c]:<10s}{base_stats[c]["r_max"]:>10.2f}{s["r_max"]:>10.2f}')
    assert s['r_max'] > base_stats[c]['r_max']
print('\n👉 关键洞察：**这个「召回天花板」有一大半不是召回问题，是分类问题**——')
print('   目标其实被检出了，只是被判进了别的类。混淆矩阵与 TIDE 分解在这里合流。')

## 8 · 工作点：给定 FP/frame 预算求 score 阈值

mAP 只用了分数的**排序**（把所有分数开平方 AP 不变），所以它对「阈值该定在哪」毫无帮助。
车端真正的需求是一个**带约束的优化**：

$$\tau^\star = \arg\max_{\tau}\ \text{Recall}(\tau) \quad \text{s.t.}\quad \frac{\text{FP}(\tau)}{N_{\text{frames}}} \le \beta$$

因为匹配本来就是按分数降序贪心做的，**按 $\tau$ 截断 = 取排序后的前缀**，
累计 FP 单调不减 ⇒ 可行集是一个前缀 ⇒ 取最后一个可行位置即最优。$O(n\log n)$ 解完。

下面对比两种做法：**全局单阈值** vs **按代价分配预算的逐类阈值**（总预算相同）。

In [ ]:
N_FRAMES = N_IMG
STATS = {c: match_class(DT, GT, c)[:2] + (int((GT['cls'] == c).sum()),) for c in range(NC)}


def pick_threshold(sc, tp, n_gt, n_frames, fp_budget_per_frame):
    # 返回 (tau, recall, fp_per_frame)；预算连一个框都放不下时返回 (None, 0, 0)
    if len(sc) == 0 or n_gt == 0:
        return None, 0.0, 0.0
    ctp = np.cumsum(tp.astype(float)); cfp = np.cumsum((~tp).astype(float))
    ok = cfp <= fp_budget_per_frame * n_frames
    if not ok.any():
        return None, 0.0, 0.0
    k = int(np.max(np.where(ok)[0]))                    # 可行集是前缀 -> 取最后一个
    return float(sc[k]), float(ctp[k] / n_gt), float(cfp[k] / n_frames)


TOTAL_BUDGET = 0.35                                     # 全系统 FP / frame

S = np.concatenate([STATS[c][0] for c in range(NC)])
T = np.concatenate([STATS[c][1] for c in range(NC)])
o = np.argsort(-S, kind='stable'); S, T = S[o], T[o]
tau_g, _, _ = pick_threshold(S, T, len(GT['img']), N_FRAMES, TOTAL_BUDGET)

print(f'【方案 A】全局单阈值 tau = {tau_g:.3f}   (总预算 {TOTAL_BUDGET} FP/frame)')
print(f'{"类别":<10s}{"recall":>9s}{"FP/frame":>11s}')
rec_g, fp_g = {}, 0.0
for c in range(NC):
    sc, tp, n_gt = STATS[c]
    m = sc >= tau_g
    rec_g[c] = float(tp[m].sum()) / n_gt
    f = float((~tp[m]).sum()) / N_FRAMES; fp_g += f
    print(f'{CLASSES[c]:<10s}{rec_g[c]:>9.2f}{f:>11.3f}')
print(f'{"合计":<10s}{"":>9s}{fp_g:>11.3f}')
assert fp_g <= TOTAL_BUDGET + 1e-9

In [ ]:
# 【方案 B】按「漏检代价」分配 FP 预算 —— 总预算完全相同
SHARE = np.array([1.0, 1.0, 1.0, 4.0, 2.0, 1.5, 0.8, 0.3])       # 停车让行 4x，指路牌 0.3x
SHARE = SHARE / SHARE.sum()

print(f'【方案 B】逐类阈值（同样的 {TOTAL_BUDGET} FP/frame 总预算，按代价重新分配）')
print(f'{"类别":<10s}{"tau":>8s}{"recall":>9s}{"vs 方案A":>10s}{"FP/frame":>11s}')
rec_p, fp_p, tau_p = {}, 0.0, {}
for c in range(NC):
    sc, tp, n_gt = STATS[c]
    tau, r, f = pick_threshold(sc, tp, n_gt, N_FRAMES, TOTAL_BUDGET * SHARE[c])
    rec_p[c], tau_p[c] = r, tau; fp_p += f
    print(f'{CLASSES[c]:<10s}{tau:>8.3f}{r:>9.2f}{r - rec_g[c]:>+10.2f}{f:>11.3f}')
print(f'{"合计":<10s}{"":>8s}{"":>9s}{"":>10s}{fp_p:>11.3f}')

assert fp_p <= TOTAL_BUDGET + 1e-9, '总预算不能超'
assert tau_p[3] < tau_g, '安全关键类的阈值必须被压低'
assert rec_p[3] > rec_g[3] and rec_p[4] > rec_g[4], '安全关键类召回必须上升'
assert rec_p[7] < rec_g[7], '代价由指路牌承担 —— 天下没有免费的召回'
print(f'\n👉 停车让行：阈值 {tau_g:.3f} -> {tau_p[3]:.3f}，召回 {rec_g[3]:.2f} -> {rec_p[3]:.2f}')
print(f'   指路牌  ：召回 {rec_g[7]:.2f} -> {rec_p[7]:.2f}（这就是代价，而且是**我们主动选的**代价）')
print('   总 FP 预算没有变。mAP 也没有变（AP 与阈值无关）—— 但产品体验完全不同。')
print('\n⚠️  这解释了一个常见事故：新模型 mAP 涨了、上车误报却暴增。')
print('   因为换 loss / 加标签平滑会整体平移分数分布，沿用旧阈值等于换了工作点。')
print('   **门禁指标必须是「工作点上的指标」，而且每换一次模型都要重解阈值。**')

## 9 · badcase 分层抽样：别让限速牌吃掉全部名额

把每个错误（含 Miss 的 GT）打上三个标签：**错误类型 × 像素尺寸 × 类别频次**，
先看看「均匀随机抽 60 张」会抽到什么。

In [ ]:
def size_bucket(box):
    s = max(box[2] - box[0], box[3] - box[1])
    return '<16px' if s < 16 else ('16-32px' if s < 32 else ('32-64px' if s < 64 else '>64px'))


FREQ = np.bincount(GT['cls'], minlength=NC) / len(GT['cls'])
def freq_bucket(c):
    return 'head' if FREQ[c] >= 0.15 else ('mid' if FREQ[c] >= 0.06 else 'tail')


RECORDS = []
for i in range(len(DT['img'])):
    if LAB[i] == 'TP':
        continue
    RECORDS.append(dict(kind='det', idx=i, err=LAB[i], size=size_bucket(DT['box'][i]),
                        freq=freq_bucket(int(DT['cls'][i])), score=float(DT['score'][i])))
for j in np.where(MISS)[0]:
    RECORDS.append(dict(kind='gt', idx=int(j), err='Miss', size=size_bucket(GT['box'][j]),
                        freq=freq_bucket(int(GT['cls'][j])), score=0.0))

STRAT = defaultdict(int)
for r in RECORDS:
    STRAT[(r['err'], r['size'], r['freq'])] += 1
print(f'badcase 池 {len(RECORDS)} 条，分成 {len(STRAT)} 层')

pick = rng.choice(len(RECORDS), 60, replace=False)
u_err = Counter(RECORDS[i]['err'] for i in pick)
print('\n均匀随机抽 60 条的错误类型分布：', dict(u_err))
print(f'其中 Bkg 占 {u_err["Bkg"] / 60:.0%}；池子里 Bkg 本来就占 '
      f'{sum(r["err"] == "Bkg" for r in RECORDS) / len(RECORDS):.0%}')
print(f'抽到的 tail 类占比 {sum(RECORDS[i]["freq"] == "tail" for i in pick) / 60:.1%}'
      f'，池子里 tail 占 {sum(r["freq"] == "tail" for r in RECORDS) / len(RECORDS):.1%}')
assert u_err['Bkg'] >= 40
print('\n⚠️  均匀抽样只是把池子的分布复制了一遍 —— 你花两小时看的全是背景误检，')
print('   而上一节刚算出背景误检的 ΔmAP 是最低的。**这不是勤奋度问题，是采样偏差问题。**')

In [ ]:
def largest_remainder(weights, total, caps):
    # 按 weights 比例把 total 个名额分给各层，受 caps 上限约束，用最大余数法保证总和精确
    keys = list(weights)
    w = np.array([weights[k] for k in keys], float)
    cap = np.array([caps[k] for k in keys], int)
    alloc = np.zeros(len(keys), int)
    remain = int(total)
    active = alloc < cap
    while remain > 0 and active.any():
        ww = np.where(active, w, 0.0)
        if ww.sum() <= 0:
            break
        raw = remain * ww / ww.sum()
        add = np.minimum(np.floor(raw).astype(int), cap - alloc)
        alloc += add; remain -= int(add.sum())
        if add.sum() == 0:                                  # 都不足 1 个 -> 按小数部分补
            for t in np.argsort(-(raw - np.floor(raw))):
                if remain <= 0:
                    break
                if active[t] and alloc[t] < cap[t]:
                    alloc[t] += 1; remain -= 1
        active = alloc < cap
    return {k: int(a) for k, a in zip(keys, alloc)}


def sqrt_allocate(sizes, n_total, n_min=1):
    # 平方根配额：n_h ∝ sqrt(N_h)，且每个非空层至少 n_min 个
    base = {k: min(n_min, v) for k, v in sizes.items()}
    used = sum(base.values())
    assert used <= n_total, '名额太少，连每层下限都凑不齐'
    caps = {k: sizes[k] - base[k] for k in sizes}
    extra = largest_remainder({k: float(np.sqrt(sizes[k])) for k in sizes}, n_total - used, caps)
    return {k: base[k] + extra[k] for k in sizes}


N_REVIEW = 120
A_sqrt = sqrt_allocate(dict(STRAT), N_REVIEW, n_min=1)
A_prop = largest_remainder({k: float(v) for k, v in STRAT.items()}, N_REVIEW, dict(STRAT))
assert sum(A_sqrt.values()) == N_REVIEW and sum(A_prop.values()) == N_REVIEW

big = sorted(STRAT, key=lambda k: -STRAT[k])[:4]
small = sorted(STRAT, key=lambda k: STRAT[k])[:4]
print(f'{"层":<34s}{"层大小":>8s}{"比例分配":>9s}{"平方根":>8s}')
for k in big + small:
    print(f'{str(k):<34s}{STRAT[k]:>8d}{A_prop[k]:>9d}{A_sqrt[k]:>8d}')

cov_p = np.mean([A_prop[k] >= 1 for k in STRAT])
cov_s = np.mean([A_sqrt[k] >= 1 for k in STRAT])
share_p = sum(v for k, v in A_prop.items() if k[0] == 'Bkg') / N_REVIEW
share_s = sum(v for k, v in A_sqrt.items() if k[0] == 'Bkg') / N_REVIEW
print(f'\n非空层被覆盖到的比例：比例分配 {cov_p:.0%}  ->  平方根+下限 {cov_s:.0%}')
print(f'Bkg 吃掉的名额比例  ：比例分配 {share_p:.0%}  ->  平方根+下限 {share_s:.0%}')
assert cov_s == 1.0 and cov_p < 0.5
assert share_s < share_p
print('\n👉 平方根压缩了层间规模差（10000 vs 100 的差距从 100 倍压到 10 倍），')
print('   再加「每层至少 1 个」的下限，就能保证**尾部类和 <16px 的层一定会被你看到**。')
print('   练习 4 会把「修复收益」也加进权重 —— 直接按 ΔmAP 分配眼睛的时间。')

## 10 · 根因决策树的代码化

把前面所有信号接进一棵树。**设计原则：先排除便宜的可能性**——
查一次类别映射 10 分钟，训一次模型 10 小时。

In [ ]:
def root_cause(m):
    # m: {'mAP', 'p10', 'gains'} -> (根因, 下一步动作, 判定依据)
    if m['mAP'] < 0.05:
        return ('管线 bug：类别 ID 偏移 / 坐标格式 / 类别表不一致',
                '转模块 03：把预测类别整体 ±1 重算 mAP；把框按 cxcywh→xyxy 转换重算 IoU 分布',
                'mAP ≈ 0')
    if m['p10'] < 0.80:
        return ('数据管线 bug：通道顺序 / 归一化 / 验证集预处理与训练不一致',
                'dump 一个 batch 的输入张量，与训练端逐元素对拍；对比 R/B 通道均值',
                'PR 曲线低召回段 precision < 0.8（最自信的预测就在错）')
    if not m['gains']:
        return ('信息不足', '先跑 TIDE 分解', '没有 ΔAP')
    top = max(m['gains'], key=lambda k: m['gains'][k])
    table = {
        'Miss': ('召回侧问题', '按像素尺寸分桶重画 PR：小目标桶趴着→C57（分辨率/P2/切片）；某类趴着→C58（长尾）'),
        'Cls':  ('分类侧问题', '看混淆矩阵找冤家对：集中在 2-3 对→两级架构 + 高分辨率 crop；弥散→类别定义有歧义'),
        'Loc':  ('定位侧问题', '看 Loc 错误的尺寸分布：集中在小目标→换尺度不敏感度量（NWD）；均匀→换 GIoU/DIoU'),
        'Both': ('定位侧问题（连锁）', '先治 Loc，Both 通常跟着降'),
        'Dupe': ('后处理问题', '查 NMS 阈值 / class-wise vs class-agnostic / 一对一分配'),
        'Bkg':  ('难负样本问题', 'OHEM / Focal Loss / 加上下文 / 提高工作点阈值'),
    }
    cause, action = table[top]
    return (cause, action, f'{top} 的 ΔAP 最大（{m["gains"][top] * 100:+.2f} pp）')


CASES = [
    ('刚接手的新代码库', dict(mAP=0.001, p10=0.00, gains={})),
    ('换了数据加载器之后', dict(mAP=0.44, p10=0.42, gains={'Cls': 0.01, 'Miss': 0.02})),
    ('本 notebook 的模型', dict(mAP=mAP0, p10=1.00, gains=GAINS)),
    ('小目标专项版本', dict(mAP=0.55, p10=0.97, gains={'Miss': 0.09, 'Cls': 0.02, 'Loc': 0.03})),
    ('NMS 阈值调错了', dict(mAP=0.61, p10=0.96, gains={'Dupe': 0.07, 'Cls': 0.02, 'Bkg': 0.01})),
]
for name, m in CASES:
    cause, action, why = root_cause(m)
    print(f'[{name}]\n  根因: {cause}\n  依据: {why}\n  动作: {action}\n')

assert root_cause(CASES[0][1])[0].startswith('管线 bug')
assert '通道顺序' in root_cause(CASES[1][1])[0]
assert root_cause(CASES[3][1])[0] == '召回侧问题'
assert root_cause(CASES[4][1])[0] == '后处理问题'
print('✅ 决策树就位。注意前两条 (mAP≈0 / p10<0.8) 都在 30 分钟内可查完，')
print('   却拦下了最浪费时间的一整类问题 —— 「训了三天发现是类别 ID 差 1」。')

## ✏️ 练习 1：TIDE 判定树

实现 `classify_fp(u_same, u_other, tf=0.5, tb=0.1)`，输入一个**非 TP** 检测的
「与同类 GT 的最大 IoU」和「与异类 GT 的最大 IoU」，返回
`'Dupe' / 'Loc' / 'Cls' / 'Both' / 'Bkg'` 之一。

**判定顺序不能变**：同类优先于异类，高 IoU 优先于低 IoU。

In [ ]:
def classify_fp(u_same, u_other, tf=0.5, tb=0.1):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
assert classify_fp(0.70, 0.00) == 'Dupe'      # 同类 IoU 够，说明 GT 被更高分的框占了
assert classify_fp(0.50, 0.00) == 'Dupe'      # 边界：>= tf
assert classify_fp(0.30, 0.00) == 'Loc'
assert classify_fp(0.10, 0.00) == 'Loc'       # 边界：>= tb
assert classify_fp(0.05, 0.80) == 'Cls'
assert classify_fp(0.05, 0.30) == 'Both'
assert classify_fp(0.02, 0.03) == 'Bkg'
assert classify_fp(0.30, 0.95) == 'Loc', '同类优先：不能因为异类 IoU 更高就判 Cls'
assert classify_fp(0.55, 0.95) == 'Dupe'
# 与完整实现对拍：所有非 TP 检测都必须一致
bad = [i for i in range(len(LAB))
       if LAB[i] != 'TP' and classify_fp(U_SAME[i], U_OTHER[i]) != LAB[i]]
assert not bad, bad[:5]
print(f'✅ 练习 1 通过：{sum(LAB != "TP")} 个非 TP 检测的判定与完整实现逐条一致。')

## ✏️ 练习 2：按性价比排序修复动作

实现 `rank_actions(counts, gains, costs)`：
- `counts[e]` 该类错误的个数，`gains[e]` 该类的 ΔmAP（小数，如 0.0875），`costs[e]` 估计工程成本（人天）
- 返回 `[(err, gain_pp, per100_pp, roi), ...]`，**按 `roi` 降序**
- `gain_pp = gains[e]*100`（百分点）；`per100_pp = gain_pp / max(counts[e],1) * 100`；`roi = gain_pp / costs[e]`

In [ ]:
def rank_actions(counts, gains, costs):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测（手算）——
c0 = {'A': 200, 'B': 10}
g0 = {'A': 0.10, 'B': 0.02}
k0 = {'A': 5.0, 'B': 0.5}
out = rank_actions(c0, g0, k0)
assert [r[0] for r in out] == ['B', 'A'], out          # roi: B=2.0/0.5=4.0 > A=10/5=2.0
assert abs(out[0][1] - 2.0) < 1e-9 and abs(out[0][2] - 20.0) < 1e-9 and abs(out[0][3] - 4.0) < 1e-9
assert abs(out[1][1] - 10.0) < 1e-9 and abs(out[1][2] - 5.0) < 1e-9 and abs(out[1][3] - 2.0) < 1e-9
# 用真实分解结果排序
real = rank_actions(CNT, GAINS, COST)
print(f'{"排名":<5s}{"类型":<7s}{"ΔmAP(pp)":>10s}{"每100个(pp)":>13s}{"ROI":>8s}')
for r, (e, gp, p100, roi) in enumerate(real, 1):
    print(f'{r:<5d}{e:<7s}{gp:>10.2f}{p100:>13.3f}{roi:>8.3f}')
assert real[0][0] == 'Cls', 'ROI 第一名应该是分类错误'
assert real[-1][0] in ('Both', 'Miss'), '成本最高/收益最低的排最后'
print('\n✅ 练习 2 通过：ΔmAP 第一名与 ROI 第一名不一定是同一个 —— 决策要用后者。')

## ✏️ 练习 3：PR 曲线诊断器

实现 `diagnose_pr(rec, prec, p_low=0.80, r_low=0.75, p_tail=0.30)`，返回
`'DATA_BUG' / 'RECALL_CEILING' / 'TAIL_COLLAPSE' / 'HEALTHY'` 之一。

**判定顺序**（先排除最便宜的可能性）：
1. `p10 < p_low` → `DATA_BUG`（最自信的预测就在错，先查管线）
2. `r_max < r_low` → `RECALL_CEILING`
3. `p50 > 0.90` 且 `p85 < p_tail` → `TAIL_COLLAPSE`
4. 否则 `HEALTHY`

可以直接复用上面的 `pr_shape_stats`。

In [ ]:
def diagnose_pr(rec, prec, p_low=0.80, r_low=0.75, p_tail=0.30):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
for k in ['HEALTHY', 'TAIL_COLLAPSE', 'DATA_BUG', 'RECALL_CEILING']:
    got = diagnose_pr(*make_curve(k))
    assert got == k, (k, got)
# 真实模型：baseline 是召回天花板，修好 Cls+Loc 之后变健康
d_base = diagnose_pr(*class_pr(DT, GT, 0))
d_fix = diagnose_pr(*class_pr(d3, g3, 0))
d_bug = diagnose_pr(*class_pr(DT_BUG, GT, 1))
print(f'限速30 baseline      -> {d_base}')
print(f'限速30 修好 Cls+Loc  -> {d_fix}')
print(f'限速60 类别整体 +1   -> {d_bug}')
assert d_base == 'RECALL_CEILING' and d_fix == 'HEALTHY' and d_bug == 'DATA_BUG'
# 空曲线（模型对该类一个框都没输出）也不能崩
assert diagnose_pr(np.zeros(0), np.zeros(0)) == 'DATA_BUG'
print('\n✅ 练习 3 通过：曲线形状 -> 病因，全程没有「看图感觉」这一步。')

## ✏️ 练习 4：按修复收益加权的分层配额

`sqrt_allocate` 已经保证了每层都被看到，但它仍然把 57% 的名额给了 ΔmAP 最低的 Bkg。
把**修复收益**也放进权重：

$$w_h = \sqrt{N_h}\;\cdot\;\max\bigl(\Delta\text{AP}_{\text{err}(h)},\ \varepsilon\bigr)$$

实现 `gain_weighted_allocate(sizes, err_gain, n_total, n_min=1, eps=1e-4)`：
- `sizes[(err, size, freq)] = N_h`，`err_gain[err] = ΔmAP`（小数）
- 每个非空层仍至少 `n_min` 个；总和精确等于 `n_total`
- 直接复用上面写好的 `largest_remainder`

In [ ]:
def gain_weighted_allocate(sizes, err_gain, n_total, n_min=1, eps=1e-4):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
A_gw = gain_weighted_allocate(dict(STRAT), GAINS, N_REVIEW, n_min=1)
assert sum(A_gw.values()) == N_REVIEW
assert all(A_gw[k] >= 1 for k in STRAT), '每个非空层至少 1 个'
assert all(A_gw[k] <= STRAT[k] for k in STRAT), '不能超过该层实际样本数'

def err_share(alloc):
    d = defaultdict(int)
    for k, v in alloc.items():
        d[k[0]] += v
    return {e: d[e] / N_REVIEW for e in ERR_TYPES}

sh_p, sh_s, sh_g = err_share(A_prop), err_share(A_sqrt), err_share(A_gw)
print(f'{"错误类型":<8s}{"ΔmAP(pp)":>10s}{"比例分配":>10s}{"平方根":>9s}{"收益加权":>10s}')
for e in ERR_TYPES:
    print(f'{e:<8s}{GAINS[e] * 100:>+10.2f}{sh_p[e]:>10.0%}{sh_s[e]:>9.0%}{sh_g[e]:>10.0%}')
assert sh_g['Bkg'] < sh_s['Bkg'] < sh_p['Bkg'], 'Bkg 的名额应被一路压下去'
assert sh_g['Cls'] > sh_s['Cls'] and sh_g['Loc'] > sh_s['Loc']
print('\n✅ 练习 4 通过：眼睛的时间也是预算，应该按 ΔmAP 分配，而不是按错误数量分配。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def classify_fp(u_same, u_other, tf=0.5, tb=0.1):
    if u_same >= tf:
        return 'Dupe'
    if u_same >= tb:
        return 'Loc'
    if u_other >= tf:
        return 'Cls'
    if u_other >= tb:
        return 'Both'
    return 'Bkg'

In [ ]:
# 练习 2 参考答案
def rank_actions(counts, gains, costs):
    out = []
    for e in gains:
        gp = gains[e] * 100.0
        out.append((e, gp, gp / max(counts.get(e, 0), 1) * 100.0, gp / costs[e]))
    out.sort(key=lambda r: -r[3])
    return out

In [ ]:
# 练习 3 参考答案
def diagnose_pr(rec, prec, p_low=0.80, r_low=0.75, p_tail=0.30):
    s = pr_shape_stats(rec, prec)
    if s['p10'] < p_low:
        return 'DATA_BUG'
    if s['r_max'] < r_low:
        return 'RECALL_CEILING'
    if s['p50'] > 0.90 and s['p85'] < p_tail:
        return 'TAIL_COLLAPSE'
    return 'HEALTHY'

In [ ]:
# 练习 4 参考答案
def gain_weighted_allocate(sizes, err_gain, n_total, n_min=1, eps=1e-4):
    base = {k: min(n_min, v) for k, v in sizes.items()}
    used = sum(base.values())
    assert used <= n_total, '名额太少，连每层下限都凑不齐'
    caps = {k: sizes[k] - base[k] for k in sizes}
    w = {k: float(np.sqrt(sizes[k])) * max(err_gain.get(k[0], 0.0), eps) for k in sizes}
    extra = largest_remainder(w, n_total - used, caps)
    return {k: base[k] + extra[k] for k in sizes}

---
## 🧪 真实工程胶囊：误差分析 SOP（可原样贴进团队 wiki）

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════════
# 检测模型误差分析 SOP —— 每训完一版就跑一遍，30 分钟出结论
# ══════════════════════════════════════════════════════════════════════

# ── 阶段 0：先证伪「管线错了」（10 分钟，永远第一步）────────────────────
#  0.1  mAP < 0.05 ?  -> 不要做误差分析，直接查三大元凶（模块 03）
#         · 预测类别整体 ±1 重算 mAP        （类别 ID 偏移 / background 占了 index 0）
#         · 框按 cxcywh->xyxy 转换重算 IoU  （坐标格式弄反）
#         · diff 训练与评测的 classes.txt   （类别表不一致）
#  0.2  PR 曲线 p(r=0.1) < 0.8 ?  -> 数据管线 bug，dump 输入张量对拍
#  通过条件：p10 >= 0.9 且 mAP 在合理量级，才允许进入阶段 1

# ── 阶段 1：TIDE 六类分解 + 修复收益（15 分钟）──────────────────────────
#  pip install tidecv     # 官方实现只有几百行，建议读一遍
#  from tidecv import TIDE, datasets
#  tide = TIDE(); tide.evaluate(datasets.COCO(gt_json), datasets.COCOResult(dt_json), mode=TIDE.BOX)
#  tide.summarize(); tide.plot()
#  产出必须包含三列：**数量 / ΔmAP / 每 100 个错误的 ΔmAP**
#  ⚠️ 只看数量必然被 Bkg 带偏（本 notebook 里 Bkg 占 87% 的数量、ΔmAP 却排倒数）
#  ⚠️ 六项 ΔAP **不可加**，别写成「Cls 贡献了 34.7% 的损失」

# ── 阶段 2：逐类 + 混淆矩阵（10 分钟）──────────────────────────────────
#  · 混淆矩阵必须是 (C+1)x(C+1)：最后一行=背景误检，最后一列=未检出
#  · 混淆矩阵**必须标注 score 阈值**，否则不可复现
#  · 逐类表三列并排：AP / GT 数 / 多种子 AP 标准差   <- 第三列防止追逐噪声
#  · 输出「最容易互相错分的 Top-5 标志对」，直接进下个迭代的 backlog

# ── 阶段 3：工作点（10 分钟）────────────────────────────────────────────
#  · 报 **Recall @ FP/frame <= beta**，不要报 precision（precision 随目标密度漂移）
#  · 逐类阈值，按漏检代价分配 FP 预算；安全关键类（停车让行/让行）单独设更低阈值
#  · **每换一次模型都要重解阈值** —— 换 loss / 加标签平滑会整体平移分数分布
#  · 单帧阈值要按「多帧确认后的 FP 预算」反推（C55 m04），否则会定得过高

# ── 阶段 4：badcase 分层抽样（人工，1-2 小时）──────────────────────────
#  分层维度 = 错误类型 x 像素尺寸(<16/16-32/32-64/>64) x 类别频次(head/mid/tail)
#  配额 = sqrt(N_h) * ΔmAP(该错误类型)，且每个非空层至少 1 个，最大余数法保证总和
#  ⚠️ 均匀随机抽样 = 把池子分布复制一遍，你会看到 90% 的背景误检
#  ⚠️ review 时必须同屏显示 GT + 预测 + 分数；10-20% 的 badcase 是**标注错误**
#      -> 界面上放一个「这是标注问题」按钮，直接回流标注团队（C58 数据闭环）
#  ⚠️ 固定一份「回归可视化集」（固定图像 ID）做版本对比 +
#      一份每次重抽的「探索集」发现新问题

# ── 阶段 5：产出物（写进实验记录，模块 01 的 schema）────────────────────
#  1. TIDE 表（数量 / ΔmAP / per-100）+ 一句话结论「下一步做 X，收益上界 +Y mAP」
#  2. 逐类表 + Top-5 混淆对
#  3. 工作点表（每类 tau / recall / FP-per-frame）
#  4. 分层 badcase 抽样清单（含 stratum id，可复现）
#  5. 本版与上一版的**同一批固定图像**的对比图
'''
print(RECIPE)
for tok in ['tidecv', '(C+1)x(C+1)', 'Recall @ FP/frame', 'sqrt(N_h)',
            '不可加', '标注错误', '重解阈值']:
    assert tok in RECIPE, tok
print('✅ SOP 覆盖：证伪管线 / 六类分解 / 逐类与混淆 / 工作点 / 分层抽样 / 产出物')

### 小结

- **mAP 只说「有多差」，不说「差在哪」**。它把六种失败压成一个标量，而这个聚合是不可逆的。
  拿到 0.60 之后你必须回到原始的匹配结果重新分桶。
- **TIDE 六类靠一棵固定的判定树定义**（同类优先、高 IoU 优先），
  分类本身不重要，重要的是**六类各对应一条不同的修复路径**。
  阈值 $t_f=0.5$、$t_b=0.1$ 与判定顺序必须写进评测配置并版本化。
- **数量最多的错误几乎从不是最值钱的**。本 notebook 里 Bkg 占错误数的 87%，
  ΔmAP 却只有 +3.05 pp（每 100 个错误 0.22 pp）；Cls 只占 2.7%，ΔmAP 却有 +8.75 pp
  （每 100 个 20.3 pp）。**原因在 AP 的定义里：FP 的杀伤力取决于它排在第几名。**
- **ΔAP 不可加**（排序/包络/求面积三重非线性），六项之和 29.7 pp ≠ 总差距 39.8 pp。
  面试里把 ΔAP 说成「可加的贡献占比」是最容易被一句话戳穿的错。
- **混淆矩阵必须是 $(C{+}1)\times(C{+}1)$**，最后一行一列信息量最大；
  而「召回天花板」经常根本不是召回问题 —— 本 notebook 里修好 Cls+Loc 之后
  各类 $r_{\max}$ 从 0.6–0.75 升到 0.8–0.92。
- **PR 曲线左端诊断「规则」，右端诊断「难度」，中段诊断「能力」。**
  $p(r{=}0.1) < 0.8$ 几乎必然是数据管线 bug —— 别调参，去查管线。
- **mAP 与阈值无关，产品与阈值强相关。** 门禁指标要用「固定 FP/frame 预算下的召回」，
  而且逐类设阈值：停车让行阈值 0.70→0.47，召回 0.54→0.65，代价由指路牌承担。
- **badcase 要分层抽样**：均匀随机抽 60 条会抽到 55 条背景误检。
  配额用 $\sqrt{N_h}\cdot\Delta\text{AP}$ + 每层下限 + 最大余数法。
- **决策树的第一原则：先排除便宜的可能性。** 查类别映射 10 分钟，训模型 10 小时。

下一站：**模块 03 · 训练与部署调试手册** —— 当阶段 0 亮红灯时，具体怎么查。